# Step 2: Build the feature table
Line up voice + face features with the engagement score, cut each recording into **1-second windows** (25 frames) and average each window into one row.
*Same as `src/02_build_features.py`; compared with the saved table instead of overwriting it.*

### Dataset
**NoXi** (NOvice eXpert Interaction) - multimodal recordings of expert-novice conversations with continuous engagement labels.
- Dataset page: https://multimediate-challenge.org/datasets/Dataset_NoXi/
- Paper: A. Cafaro et al., "The NoXi Database: Multimodal Recordings of Mediated Novice-Expert Interactions", ACM ICMI 2017, https://doi.org/10.1145/3136755.3136780
- Used here: 9 conversations (English 027, 028, 030; French 001, 013, 021; German 070, 072, 077) from the NoXi train set:
  eGeMAPS voice features, OpenFace 2.0 face features, engagement labels (25 per second) and the pretrained-Transformer embeddings shipped with the corpus.
- Licence: NoXi is available for research under an End-User Licence Agreement (signed by the supervisor). The data is **not included** here and must not be shared;
  request access on the dataset page and place the downloaded zips as described in `01_extract_data`.

Go to the project folder.

In [ ]:
import os
if os.getcwd().endswith("notebooks"):
    os.chdir("..")          # work from the project folder

In [ ]:
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

**Read a feature file.** The `.stream` header says how many numbers each frame has; the `.stream~` file holds the numbers.

In [ ]:
def read_stream(path):
    info = ET.parse(path).getroot().find("info")
    dim = int(info.get("dim"))                       # numbers per frame
    if info.get("type") == "FLOAT":
        data = np.fromfile(path + "~", dtype=np.float32)
    else:
        data = np.fromfile(path + "~", dtype=np.float64)
    frames = data.size // dim
    return data[: frames * dim].reshape(frames, dim)     # table: frames x numbers

**Read the labels** (bad values become NaN) **and the language**.

In [ ]:
def read_labels(path):
    values = pd.read_csv(path, header=None)[0]
    return pd.to_numeric(values, errors="coerce").to_numpy("float32")


def read_language(folder):
    text = open(folder + "/language.annotation.csv").read()     # e.g. 0.00;904.00;French;1.00
    return text.strip().split(";")[2]

**Main loop:** every session, both people, every 1-second window.

In [ ]:
rows = []         # one row per window (averaged)
sequences = []    # the 25 raw frames of each window (for CNN / RNN / Transformer)

for session in sorted(os.listdir("data/noxi")):
    folder = "data/noxi/" + session
    language = read_language(folder)
    for role in ["expert", "novice"]:
        voice = read_stream(folder + "/" + role + ".audio.egemapsv2.stream")
        face = read_stream(folder + "/" + role + ".openface2.stream")
        label = read_labels(folder + "/" + role + ".engagement.annotation.csv")

        n = min(len(voice), len(face), len(label))      # cut all three to the same length
        for w in range(n // 25):
            start, end = w * 25, (w + 1) * 25           # the 25 frames of second w
            engagement = np.nanmean(label[start:end])
            if np.isnan(engagement):                    # no valid label: skip this second
                continue
            row = {}
            voice_mean = np.nanmean(voice[start:end], axis=0)
            face_mean = np.nanmean(face[start:end], axis=0)
            for i in range(len(voice_mean)):
                row["voice_" + str(i)] = float(voice_mean[i])
            for i in range(len(face_mean)):
                row["face_" + str(i)] = float(face_mean[i])
            row["engagement"] = float(engagement)
            row["language"] = language
            row["session"] = session
            row["role"] = role
            rows.append(row)

            frames = np.concatenate([voice[start:end], face[start:end]], axis=1)
            sequences.append(np.nan_to_num(frames).astype(np.float32))
    print(session, language, "done")

df = pd.DataFrame(rows)
seq = np.stack(sequences)
print(df.shape, seq.shape)

**Windows per language**

In [ ]:
print(df["language"].value_counts())

**Same as the files saved by the script?** (should print True, True)

In [ ]:
saved = pd.read_parquet("data/features/noxi_engagement.parquet")
saved_seq = np.load("data/features/noxi_sequences.npy")
print(np.allclose(saved.select_dtypes("number"), df.select_dtypes("number"), equal_nan=True))
print(np.array_equal(saved_seq, seq))